In [15]:
import os
import pandas as pd
import kagglehub
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


path = kagglehub.dataset_download("srisyra02/online-food-ordering-dataset")
csv_path = os.path.join(path, "online food delivery dataset.csv")
df = pd.read_csv(csv_path)
df.head()

,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,Customer Type,latitude,longitude,Pin code,Output,Feedback,Unnamed: 13
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,560001,Yes,Positive,Yes
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,560009,Yes,Positive,Yes
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,560017,Yes,Negative,Yes
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,560019,Yes,Positive,Yes
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,560010,Yes,Positive,Yes


In [16]:
df['Monthly Income'].unique()
Monthly_Income_encode_mapping={'No Income':0, 'Below Rs.10000':1, 'More than 50000':4, '10001 to 25000':3,
       '25001 to 50000':2}
df['modi_Monthly Income']=df['Monthly Income'].map(Monthly_Income_encode_mapping)


In [17]:
df['Educational Qualifications'].unique() 
Educational_Qualifications_encode_mapping={'Post Graduate':3, 'Graduate':2, 'Ph.D':4, 'Uneducated':0, 'School':1}
df['modi_Educational Qualifications']=df['Educational Qualifications'].map(Educational_Qualifications_encode_mapping)


In [18]:
df['Customer Type'].unique()
Customer_Type_encode_mapping={'Frequent':0, 'Regular':2, 'New':1}
df['modi_Customer Type']=df['Customer Type'].map(Customer_Type_encode_mapping)


In [19]:
df['Feedback'].unique()
Feedback_encode_mapping={'Positive':1, 'Negative':0}
df['modi_Feedback']=df['Feedback'].map(Feedback_encode_mapping)

In [20]:
df.columns

Index(['Age', 'Gender', 'Marital Status', 'Occupation', 'Monthly Income',
       'Educational Qualifications', 'Family size', 'Customer Type',
       'latitude', 'longitude', 'Pin code', 'Output', 'Feedback',
       'Unnamed: 13', 'modi_Monthly Income', 'modi_Educational Qualifications',
       'modi_Customer Type', 'modi_Feedback'],
      dtype='object')

In [21]:
df.describe()

,Age,Family size,latitude,longitude,Pin code,modi_Monthly Income,modi_Educational Qualifications,modi_Customer Type,modi_Feedback
count,388.000000,388.000000,388.000000,388.000000,388.000000,388.000000,388.000000,388.000000,317.0
mean,24.628866,3.280928,12.972058,77.600160,560040.113402,1.407216,2.525773,1.185567,1.0
std,2.975593,1.351025,0.044489,0.051354,31.399609,1.549910,0.679607,0.951864,0.0
min,18.000000,1.000000,12.865200,77.484200,560001.000000,0.000000,0.000000,0.000000,1.0
25%,23.000000,2.000000,12.936900,77.565275,560010.750000,0.000000,2.000000,0.000000,1.0
50%,24.000000,3.000000,12.977000,77.592100,560033.500000,1.000000,3.000000,2.000000,1.0
75%,26.000000,4.000000,12.997025,77.630900,560068.000000,3.000000,3.000000,2.000000,1.0
max,33.000000,6.000000,13.102000,77.758200,560109.000000,4.000000,4.000000,2.000000,1.0


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 388 entries, 0 to 387
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Age                              388 non-null    int64  
 1   Gender                           388 non-null    object 
 2   Marital Status                   388 non-null    object 
 3   Occupation                       388 non-null    object 
 4   Monthly Income                   388 non-null    object 
 5   Educational Qualifications       388 non-null    object 
 6   Family size                      388 non-null    int64  
 7   Customer Type                    388 non-null    object 
 8   latitude                         388 non-null    float64
 9   longitude                        388 non-null    float64
 10  Pin code                         388 non-null    int64  
 11  Output                           388 non-null    object 
 12  Feedback              

In [23]:
print(df.isnull().sum())

Age                                 0
Gender                              0
Marital Status                      0
Occupation                          0
Monthly Income                      0
Educational Qualifications          0
Family size                         0
Customer Type                       0
latitude                            0
longitude                           0
Pin code                            0
Output                              0
Feedback                            0
Unnamed: 13                         0
modi_Monthly Income                 0
modi_Educational Qualifications     0
modi_Customer Type                  0
modi_Feedback                      71
dtype: int64


In [24]:
print(df[df.duplicated()])

     Age  Gender Marital Status Occupation  Monthly Income  \
43    22  Female         Single    Student       No Income   
48    23  Female         Single    Student       No Income   
49    22  Female         Single    Student  10001 to 25000   
52    24    Male         Single    Student       No Income   
60    23    Male         Single    Student       No Income   
..   ...     ...            ...        ...             ...   
382   24  Female         Single    Student       No Income   
383   23  Female         Single    Student       No Income   
384   23  Female         Single    Student       No Income   
385   22  Female         Single    Student       No Income   
387   23    Male         Single    Student       No Income   

    Educational Qualifications  Family size Customer Type  latitude  \
43               Post Graduate            5      Frequent   12.9850   
48                    Graduate            3       Regular   12.9820   
49               Post Graduate            

In [25]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

In [26]:
feature_name=['Age', 'Gender', 'Marital Status', 'Occupation', 
    'modi_Monthly Income', 'modi_Educational Qualifications', 'Family size', 
    'modi_Customer Type', 'latitude', 'longitude', 'Pin code']
df_clean=df.dropna(subset=feature_name+['modi_Feedback'])
df_clean=df.drop_duplicates().copy()
df_clean['modi_Output'] = df_clean['Output'].map({'Yes': 1, 'No': 0})
X=df_clean[feature_name]
y=df_clean['modi_Output']
X_encoded = pd.get_dummies(X, drop_first=True)
X_train,X_test,y_train,y_test=train_test_split(X_encoded,y,random_state=1)

In [27]:
X_encoded.head()

,Age,modi_Monthly Income,modi_Educational Qualifications,Family size,modi_Customer Type,latitude,longitude,Pin code,Gender_Male,Marital Status_Prefer not to say,Marital Status_Single,Occupation_House wife,Occupation_Self Employeed,Occupation_Student
0,20,0,3,4,0,12.9766,77.5993,560001,False,False,True,False,False,True
1,24,1,2,3,2,12.9770,77.5773,560009,False,False,True,False,False,True
2,22,1,3,3,2,12.9551,77.6593,560017,True,False,True,False,False,True
3,22,0,2,6,0,12.9473,77.5616,560019,False,False,True,False,False,True
4,22,1,3,4,0,12.9850,77.5533,560010,True,False,True,False,False,True


In [30]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
KNN=KNeighborsClassifier(n_neighbors=5)
KNN.fit(X_train,y_train)

KNeighborsClassifier()

In [29]:
KNN.score(X_test, y_test)

0.7916666666666666